# COMP9444 Project 005 — GRU Baseline and Optimization
## Pedestrian Trajectory Prediction in Crowds

This notebook presents the completed GRU experiments on the Grand Central (GC) pedestrian dataset, including the baseline, evaluation, turning-trajectory analysis, and optimization.

The final team project compares three motion encoders: **GRU, LSTM, and Transformer**. LSTM and Transformer results are left as placeholders until the team results are available.


## 1. Project Paths

Absolute paths are retained from the working notebook so the existing cache and trained checkpoints can be reused.


In [ ]:
from pathlib import Path
import os

HERE = Path.cwd().resolve()

# Automatically locate gc_data_interface.
# Supports both:
# 1) notebook in .../project/
# 2) notebook in .../9444_Proj/gc_data_interface/gru/
candidates = [
    HERE,
    HERE.parent,
    HERE / "gc_data_interface",
    HERE.parent / "gc_data_interface",
    HERE / "9444_Proj" / "gc_data_interface",
    HERE.parent / "9444_Proj" / "gc_data_interface",
]

CODE_ROOT = None
for p in candidates:
    if (p / "team_train.py").exists():
        CODE_ROOT = p.resolve()
        break

if CODE_ROOT is None:
    raise FileNotFoundError(
        "Could not find gc_data_interface/team_train.py. "
        "Open the notebook from the project folder or "
        "gc_data_interface/gru/."
    )

# Dataset: environment variable first, then common nearby locations.
dataset_candidates = []

if "COMP9444_DATASET_ROOT" in os.environ:
    dataset_candidates.append(Path(os.environ["COMP9444_DATASET_ROOT"]))

dataset_candidates += [
    CODE_ROOT.parent / "cvpr2015_pedestrianWalkingPathDataset",
    CODE_ROOT.parent.parent / "cvpr2015_pedestrianWalkingPathDataset",
    HERE / "cvpr2015_pedestrianWalkingPathDataset",
    HERE.parent / "cvpr2015_pedestrianWalkingPathDataset",
]

DATASET_ROOT = None
for p in dataset_candidates:
    if p.exists():
        DATASET_ROOT = p.resolve()
        break

CACHE_DIR = CODE_ROOT / "work" / "team_cache"
RUNS_DIR = CODE_ROOT / "runs"

print("HERE         =", HERE)
print("CODE_ROOT    =", CODE_ROOT)
print("DATASET_ROOT =", DATASET_ROOT)
print("CACHE_DIR    =", CACHE_DIR)
print("RUNS_DIR     =", RUNS_DIR)

if DATASET_ROOT is None:
    print(
        "\nDataset not found automatically. "
        "Set COMP9444_DATASET_ROOT to your dataset folder before "
        "running data preparation/training."
    )


## 2. Dataset and Data Split

The Grand Central pedestrian dataset is used for pedestrian trajectory prediction.

| Split | Samples |
|---|---:|
| Train | 203,475 |
| Validation | 43,814 |
| Test | 45,454 |

The same prepared data and evaluation protocol are used across the team models to support a fair comparison.


## 3. Data Preparation

The preprocessing pipeline prepares heading-aligned motion information together with spatial and social context. The dataset cache has already been generated, so preprocessing is disabled by default below.


In [ ]:
import subprocess

RUN_PREPARE = False

if RUN_PREPARE:
    subprocess.run([
        "python3", str(CODE_ROOT / "team_prepare.py"),
        "--root", str(DATASET_ROOT),
        "--cache-dir", str(CACHE_DIR),
    ], check=True)
else:
    print("Skipped — data preparation was already completed.")

## 4. Models and Methods

### 4.1 Shared Forecasting Framework

**Observed trajectory → Motion encoder → Motion representation → Spatial + social features → Feature fusion → Waypoint/endpoint prediction → Future trajectory**

The main architectural choice changed between team runs is the motion encoder.

### 4.2 GRU Motion Encoder

The project uses PyTorch `nn.GRU` from `motion_encoders.py`.

| Parameter | Value |
|---|---:|
| Input size | 4 |
| Hidden size | 64 |
| Recurrent layers | 1 |
| Batch first | True |
| Spatial features | Enabled |
| Social features | Enabled |

The final GRU hidden state is combined with spatial and social features by the shared forecasting model. A manually implemented `CustomGRUCell` is not used in the experiment.


In [ ]:
import torch
import torch.nn as nn

gru_reference = nn.GRU(
    input_size=4,
    hidden_size=64,
    num_layers=1,
    batch_first=True,
    dropout=0.0
)
print(gru_reference)


## 5. Experimental Setup

### Training Configuration

| Parameter | Value |
|---|---:|
| Seed | 42 |
| Hidden size | 64 |
| Recurrent layers | 1 |
| Batch size | 256 |
| Epochs | 16 |
| Learning rate | 0.001 |
| Minimum learning rate | 0.0002 |
| Weight decay | 0.0001 |
| Gradient clip | 5.0 |
| Encoder dropout setting | 0.1 |
| Baseline turn weight | 1.0 |
| Patience | 5 |
| Minimum epochs | 10 |

Because the GRU has only one recurrent layer, PyTorch does not apply recurrent dropout between recurrent layers. The complete forecasting model contains **80,449 parameters**.

### Evaluation Metrics

**ADE** is the average Euclidean distance between predicted and ground-truth future positions.

\[
ADE=\frac{1}{T}\sum_{t=1}^{T}\|\hat{p}_t-p_t\|_2
\]

**FDE** is the Euclidean distance between the final predicted and ground-truth positions.

\[
FDE=\|\hat{p}_T-p_T\|_2
\]

Lower values are better.


## 6. GRU Smoke Test

A small test was first run using 2,048 training samples, 512 validation samples, and 2 epochs to verify that the training pipeline worked correctly.


In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    subprocess.run([
        "python3", str(CODE_ROOT / "team_train.py"),
        "--root", str(DATASET_ROOT),
        "--encoder", "gru",
        "--cache-dir", str(CACHE_DIR),
        "--output", str(RUNS_DIR / "gru_test"),
        "--epochs", "2",
        "--train-limit", "2048",
        "--val-limit", "512",
    ], check=True)
else:
    print("Skipped — smoke test was already completed.")

### Smoke Test Results

| Epoch | Loss | ADE (px) | FDE (px) |
|---|---:|---:|---:|
| 1 | 0.2973 | 28.12 | 51.92 |
| 2 | 0.2914 | 27.71 | 51.09 |

The decreasing errors confirmed that the GRU training pipeline worked correctly.


## 7. Full GRU Training

The GRU was then trained on the full training set for up to 16 epochs. The best checkpoint was saved as `runs/gru/best.pt`.

Training is disabled by default because it has already been completed.


In [ ]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    subprocess.run([
        "python3", str(CODE_ROOT / "team_train.py"),
        "--root", str(DATASET_ROOT),
        "--encoder", "gru",
        "--cache-dir", str(CACHE_DIR),
        "--output", str(RUNS_DIR / "gru"),
    ], check=True)
else:
    print("Skipped — full GRU training was already completed.")
    print(f"Checkpoint: {RUNS_DIR / 'gru' / 'best.pt'}")

## 8. Validation Results

| Metric | Result |
|---|---:|
| Training Loss | 0.1661 |
| Validation ADE | 19.98 px |
| Validation FDE | 34.50 px |
| Turn ADE | 31.27 px |
| Turn FDE | 59.98 px |

Turning trajectories already show larger errors than the overall validation set.


## 9. Test Evaluation

The saved baseline checkpoint was evaluated on the complete test split. Evaluation is disabled below because `runs/gru/test.json` has already been generated.


In [ ]:
RUN_TEST_EVAL = False

if RUN_TEST_EVAL:
    subprocess.run([
        "python3", str(CODE_ROOT / "team_eval.py"),
        "--root", str(DATASET_ROOT),
        "--cache-dir", str(CACHE_DIR),
        "--checkpoint", str(RUNS_DIR / "gru" / "best.pt"),
        "--split", "test",
        "--output", str(RUNS_DIR / "gru" / "test.json"),
    ], check=True)
else:
    print("Skipped — test evaluation was already completed.")
    print(f"Result: {RUNS_DIR / 'gru' / 'test.json'}")

## 10. GRU Baseline Test Results

| Metric | Result |
|---|---:|
| Test Samples | 45,454 |
| **ADE ↓** | **21.59 px** |
| **FDE ↓** | **37.34 px** |
| Turn Samples | 8,309 |
| **Turn ADE ↓** | **32.11 px** |
| **Turn FDE ↓** | **61.37 px** |

The much larger turning errors identify the main weakness of the baseline GRU.


In [ ]:
baseline_results = {
    "Test Samples": 45454,
    "ADE (px)": 21.590505599975586,
    "FDE (px)": 37.33546447753906,
    "Turn Samples": 8309,
    "Turn ADE (px)": 32.10686111450195,
    "Turn FDE (px)": 61.37189483642578,
    "Parameters": 80449
}
for k, v in baseline_results.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


### 10.1 Overall vs Turning Error


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["ADE", "FDE"]
overall = [21.5905, 37.3355]
turning = [32.1069, 61.3719]
x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(7,5))
b1 = plt.bar(x-width/2, overall, width, label="Overall")
b2 = plt.bar(x+width/2, turning, width, label="Turning")
plt.xlabel("Metric")
plt.ylabel("Error (pixels)")
plt.title("GRU Baseline: Overall vs Turning Error")
plt.xticks(x, metrics)
plt.legend()
for bars in (b1,b2):
    for bar in bars:
        h=bar.get_height()
        plt.text(bar.get_x()+bar.get_width()/2,h+1,f"{h:.2f}",ha="center")
plt.tight_layout()
plt.show()


## 11. GRU Optimization — Turning Trajectories

### 11.1 Motivation

The baseline GRU performs noticeably worse on turning trajectories. To reduce this weakness, the training loss weight for turning samples was increased so that turning errors contribute more strongly during training.

### 11.2 Ablation Setup

Three settings were tested:

- `turn_weight = 1.00` — baseline
- `turn_weight = 1.25`
- `turn_weight = 1.50`

All other architecture and training settings were kept unchanged.


### 11.3 Optimization Results

| Turn Weight | Overall ADE ↓ | Overall FDE ↓ | Turn ADE ↓ | Turn FDE ↓ |
|---:|---:|---:|---:|---:|
| 1.00 | **21.59** | **37.34** | 32.11 | 61.37 |
| **1.25** | 21.61 | 37.35 | 31.55 | 59.91 |
| 1.50 | 21.77 | 37.74 | **31.24** | **59.43** |

Increasing turn weight improves turning-specific prediction, but a larger weight introduces a trade-off with overall performance.


In [ ]:
optimization_results = {
    "Baseline (1.00)": [21.5905, 37.3355, 32.1069, 61.3719],
    "Turn Weight 1.25": [21.6064, 37.3451, 31.5508, 59.9101],
    "Turn Weight 1.50": [21.7654, 37.7415, 31.2404, 59.4286]
}
for name, vals in optimization_results.items():
    print(name)
    print(f"  ADE: {vals[0]:.2f} px")
    print(f"  FDE: {vals[1]:.2f} px")
    print(f"  Turn ADE: {vals[2]:.2f} px")
    print(f"  Turn FDE: {vals[3]:.2f} px\n")


### 11.4 Turning Performance Comparison


In [ ]:
models = ["Baseline\n1.00","Turn Weight\n1.25","Turn Weight\n1.50"]
turn_ade=[32.1069,31.5508,31.2404]
turn_fde=[61.3719,59.9101,59.4286]
x=np.arange(len(models)); width=.35
plt.figure(figsize=(8,5))
b1=plt.bar(x-width/2,turn_ade,width,label="Turn ADE")
b2=plt.bar(x+width/2,turn_fde,width,label="Turn FDE")
plt.xlabel("GRU Configuration"); plt.ylabel("Error (pixels)")
plt.title("Effect of Turn Weight on Turning Prediction")
plt.xticks(x,models); plt.legend()
for bars in (b1,b2):
    for bar in bars:
        h=bar.get_height()
        plt.text(bar.get_x()+bar.get_width()/2,h+.4,f"{h:.2f}",ha="center")
plt.tight_layout(); plt.show()


### 11.5 Overall Performance Comparison


In [ ]:
overall_ade=[21.5905,21.6064,21.7654]
overall_fde=[37.3355,37.3451,37.7415]
x=np.arange(len(models)); width=.35
plt.figure(figsize=(8,5))
b1=plt.bar(x-width/2,overall_ade,width,label="Overall ADE")
b2=plt.bar(x+width/2,overall_fde,width,label="Overall FDE")
plt.xlabel("GRU Configuration"); plt.ylabel("Error (pixels)")
plt.title("Effect of Turn Weight on Overall GRU Performance")
plt.xticks(x,models); plt.legend()
for bars in (b1,b2):
    for bar in bars:
        h=bar.get_height()
        plt.text(bar.get_x()+bar.get_width()/2,h+.25,f"{h:.2f}",ha="center")
plt.tight_layout(); plt.show()


### 11.6 Selected Optimized GRU

`turn_weight = 1.25` is selected as the optimized GRU.

Compared with the baseline:

- Turn ADE: **32.11 → 31.55 px**
- Turn FDE: **61.37 → 59.91 px**
- Overall ADE: **21.59 → 21.61 px**
- Overall FDE: **37.34 → 37.35 px**

The 1.50 setting achieves the best turning metrics but produces a larger degradation in overall performance. Therefore, 1.25 provides the better balance.


In [ ]:
turn_ade_improvement=(32.1069-31.5508)/32.1069*100
turn_fde_improvement=(61.3719-59.9101)/61.3719*100
overall_ade_change=(21.6064-21.5905)/21.5905*100
overall_fde_change=(37.3451-37.3355)/37.3355*100

print(f"Turn ADE improvement: {turn_ade_improvement:.2f}%")
print(f"Turn FDE improvement: {turn_fde_improvement:.2f}%")
print(f"Overall ADE change: +{overall_ade_change:.2f}%")
print(f"Overall FDE change: +{overall_fde_change:.2f}%")


## 12. Trajectory Visualization / Error Analysis

Typical, best, and difficult prediction examples were explored during development.

Before using the preliminary trajectory plots as final report figures, the conversion between local trajectory coordinates and absolute image coordinates should be verified. A possible visual discontinuity was observed between the history and future portions of the preliminary plots.

The quantitative ADE/FDE results above remain the primary evaluation evidence until the plotting coordinates are confirmed.


## 13. Team Model Comparison

| Model | ADE ↓ | FDE ↓ | Turn ADE ↓ | Turn FDE ↓ |
|---|---:|---:|---:|---:|
| GRU Baseline | 21.59 | 37.34 | 32.11 | 61.37 |
| **Optimized GRU (1.25)** | **21.61** | **37.35** | **31.55** | **59.91** |
| LSTM | TBD | TBD | TBD | TBD |
| Transformer | TBD | TBD | TBD | TBD |

LSTM and Transformer values should be added only after those team experiments are completed under the same evaluation protocol.


## 14. Discussion

### Strengths
The GRU is a compact recurrent encoder and achieves stable overall trajectory prediction performance within the shared spatial/social forecasting framework.

### Weakness
The main weakness is turning trajectories. The baseline Turn ADE and Turn FDE are substantially higher than the overall metrics.

### Effect of Optimization
Increasing the turning-sample weight improves turning prediction. The 1.25 setting improves both Turn ADE and Turn FDE while leaving overall performance almost unchanged.

### Limitation
The optimization reduces but does not eliminate the turning-trajectory problem. Even after optimization, turning errors remain higher than overall errors.

### Future Work
Future work could investigate stronger modelling of direction changes, improved social-interaction modelling, additional GRU architecture experiments, and comparison with the LSTM and Transformer encoders.


## 15. Conclusion

The baseline GRU achieved **21.59 px ADE** and **37.34 px FDE** on the GC test set.

Turning trajectories were identified as the main weakness. Increasing the turn weight from 1.00 to 1.25 reduced Turn ADE from **32.11 px to 31.55 px** and Turn FDE from **61.37 px to 59.91 px**, while overall ADE and FDE remained almost unchanged.

Therefore, `turn_weight = 1.25` is selected as the optimized GRU configuration. The final team analysis should compare this model with LSTM and Transformer under the same evaluation protocol.
